# Gold ingest


In [0]:
df_orders=spark.read.format("delta").load("/Volumes/my_local_data/silver/retail_sales/orders")
df_customers=spark.read.format("delta").load("/Volumes/my_local_data/silver/retail_sales/customers")

df_sales = df_orders.alias("orders").join(df_customers.alias("customers"), df_orders.customer_id == df_customers.customer_id, "inner").select('customers.customer_id','name','city','signup_date','order_id','order_date','amount','payment_method','orders.ingestion_date')

df_sales_daily=df_sales.groupBy('order_date').sum('amount').withColumnRenamed('sum(amount)','total_sales')

df_sales_customer = df_sales.groupBy("customer_id", "name") \
                                  .sum("amount") \
                                  .withColumnRenamed("sum(amount)", "total_sales")

df_sales_location = df_sales.groupBy("city") \
                                  .sum("amount") \
                                  .withColumnRenamed("sum(amount)", "total_sales")


df_sales.write.format("delta").mode("overwrite").save("/Volumes/my_local_data/gold/retail_sales/sales")

df_sales_daily.write.format("delta").mode("overwrite").save("/Volumes/my_local_data/gold/retail_sales/sales_daily")

df_sales_customer.write.format("delta").mode("overwrite").save("/Volumes/my_local_data/gold/retail_sales/sales_customer")

df_sales_location.write.format("delta") .mode("overwrite").save("/Volumes/my_local_data/gold/retail_sales/sales_location")

#tables

df_sales.write.format("delta").mode("overwrite").saveAsTable("my_local_data.gold.sales")

df_sales_daily.write.format("delta").mode("overwrite").saveAsTable("my_local_data.gold.sales_daily")

df_sales_customer.write.format("delta").mode("overwrite").saveAsTable("my_local_data.gold.sales_customer")

df_sales_location.write.format("delta") .mode("overwrite").saveAsTable("my_local_data.gold.sales_location")

print("Gold ingestion is successful")
print("Final tables are created as per business logic")



In [0]:
%sql
select * from my_local_data.gold.sales;
select * from my_local_data.gold.sales_location;